In [3]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp

In [4]:
# Base test case: A simple nonlinear system
def func(x):
    return np.array([
        x[0]**2 + x[1]**2 - 4,
        x[1] - x[0]**2 + 1
    ])

def jac(x):
    return np.array([
        [2*x[0], 2*x[1]],
        [-2*x[0], 1]
    ])


In [7]:
# Test local methods
import newton_local_deuflhard as local_d
import newton_local as local

x0 = np.array([0.5, 0.5])

success_d, x_d = local_d.solve(
    func, jac, x0, XTOL = 1e-6, MAX_ITER = 20
)

success_c, x_c = local.solve(
    func, jac, x0, RTOL = 1e-6, MAX_ITER = 20
)


print("Deuflhard's method:")
print(f"Success: {success_d}, Solution: {x_d}")

print("Classical Newton's method:")
print(f"Success: {success_c}, Solution: {x_c}")

Converged at step 5
Converged at step 6
Deuflhard's method:
Success: True, Solution: [1.51748991 1.30277564]
Classical Newton's method:
Success: True, Solution: [1.51748991 1.30277564]


In [ ]:
x0 = np.array([0.5, 0.5])
step_0 = 1.0
MAX_NR_ITER = 20
MAX_LS_ITER = 10
XTOL = 1e-6
step_min = 1e-8

In [ ]:
x = x0.copy()
norm_dx = 0.0
norm_dx_prev = 0.0
norm_dx_ls = 0.0
dx_prev = np.zeros_like(x)
dx_ls = np.zeros_like(x)
step = step_0



for k in range(MAX_NR_ITER):
    F = func(x)
    J = jac(x)
    dx = np.linalg.solve(J, -F)
    norm_dx = np.linalg.norm(dx)
    if norm_dx < XTOL:
        print(f"Converged in {k+1} iterations.")
        break

    # Line search
    if k==0:
        step = step_0
    else:
        mu = norm_dx_prev * norm_dx_ls / np.linalg.norm(dx_ls - dx) / norm_dx * step
        step = np.min([1.0, mu])
        if step < step_min:
            print("Line search step too small. Stopping.")
            break

    step_found = False
    for i in range(MAX_LS_ITER):
        x_ls = x + step * dx
        F = func(x_ls)
        J = jac(x_ls)
        dx_ls = np.linalg.solve(J, -F)
        norm_dx_ls = np.linalg.norm(dx_ls)
        Theta = norm_dx_ls / norm_dx
        mu = 0.5 * step * step * norm_dx / np.linalg.norm(dx_ls - (1 - step) * dx)

        if Theta > 1 - 0.5 * step:
            step = np.min([step * 0.5, mu])
            if step < step_min:
                print("Line search step too small. Stopping.")
                break
            else:
                continue
        else:
            step_temp = np.min([1.0, mu])
            if (step == 1.0) and (step_temp == 1.0):
                if norm_dx_ls < XTOL:
                    x = x_ls.copy()
                    print(f"Converged in {k+1} iterations.")
                    break
                elif Theta < 0.5:
                    # switch to a local method, to be implemented
                    pass
            elif step_temp >= 4 * step:
                step = step_temp
                break
            else:
                step_found = True
                x = x_ls.copy()

    if not step_found:
        print(f"Line search failed after {MAX_LS_ITER} iterations. Stopping.")
        break    

    norm_dx_prev = norm_dx